In [1]:
# Dataset manipulation
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import category_encoders as ce
from optbinning import OptimalBinning

# Models
from sklearn.linear_model import LogisticRegression

# Metrics
from sklearn.metrics import recall_score, classification_report

In [2]:
# read data
data_path = '../data/diabetes_prediction_dataset.csv'
data = pd.read_csv(data_path, sep=',')
print('Filas del dataset en bruto:', data.shape[0])
# Filter data
data = data[data['bmi'] < 65]
data = data[data['gender']!='Other']
print('Filas del dataset después de filtrar:', data.shape[0])
# check data
data.head()

Filas del dataset en bruto: 100000
Filas del dataset después de filtrar: 99935


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [3]:
# Features and target
X = data.drop(columns=["diabetes"])
y = data["diabetes"]   # binary: 0/1

# Split first (important to avoid leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
# Custom transformer for optimal binning
class MultiOptimalBinningWOE(BaseEstimator, TransformerMixin):
    def __init__(self, binning_config):
        self.binning_config = binning_config
        self.binners = {}

    def fit(self, X, y):
        self.feature_names_in_ = list(X.columns)  # IMPORTANT
        X = X.copy()

        for col, max_bins in self.binning_config.items():
            optb = OptimalBinning(
                name=col,
                dtype="numerical",
                max_n_bins=max_bins
            )
            optb.fit(X[col], y)
            self.binners[col] = optb

        return self

    def transform(self, X):
        X = X.copy()

        for col, optb in self.binners.items():
            X[col] = optb.transform(X[col])

        return X
    
    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_

# Custom transformer for type casting
class TypeCaster(BaseEstimator, TransformerMixin):
    def __init__(self, columns, dtype):
        self.columns = columns
        self.dtype = dtype

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.columns] = X[self.columns].astype(self.dtype)
        return X

In [13]:
# Declare columns
binary_cols = ['hypertension','heart_disease']
woe_cat_cols = ['gender','smoking_history','hypertension','heart_disease']

# Numerical columns to bin and their max number of bins
binning_config = {
    "blood_glucose_level": 4,
    'HbA1c_level': 5,
    "age": 8,
    "bmi": 8

}

# Transformations
binary_to_cat = TypeCaster(
    columns=binary_cols,
    dtype=str
)

# Preprocessor
preprocessor = ColumnTransformer([
    ("woe", ce.WOEEncoder(), woe_cat_cols),
    ("bin_woe",MultiOptimalBinningWOE(binning_config),list(binning_config.keys()))
    ],
remainder="passthrough",
verbose_feature_names_out=True
)

# Pipeline
log_pipe_line = Pipeline([
    ("type_cast", binary_to_cat),
    ("preprocessor", preprocessor),
])

log_pipe_line.fit(X_train, y_train)

feature_names = log_pipe_line.named_steps["preprocessor"].get_feature_names_out()

df_processed = log_pipe_line.transform(X_train)
df_processed = pd.DataFrame(df_processed,columns=feature_names)
df_processed.head()

,woe__gender,woe__smoking_history,woe__hypertension,woe__heart_disease,bin_woe__blood_glucose_level,bin_woe__HbA1c_level,bin_woe__age,bin_woe__bmi
0,0.149763,0.223560,-0.225959,-0.131395,1.612593,1.809661,0.353835,1.079046
1,-0.118977,-0.774097,-0.225959,-0.131395,1.612593,1.809661,4.200568,2.398289
2,-0.118977,0.118364,1.438288,-0.131395,0.199551,0.094640,-0.111718,0.295259
3,0.149763,0.118364,-0.225959,-0.131395,-1.809115,0.094640,2.168497,0.295259
4,-0.118977,0.118364,-0.225959,-0.131395,-1.809115,0.094640,1.125706,0.295259


#### Check the sense of the woe, if this is the same in optimal binning that in the woe_encoder.

Once that is ok we can adjust the logistic regresion.